# WavLM Utterance-Level Extraction

Extracts embeddings for all 239K utterances from 645 audio files.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/gdrive')
print('Drive mounted!')

In [ ]:
# Install dependencies
!pip install -q transformers librosa
import torch
print(f'PyTorch: {torch.__version__}')

In [ ]:
# Load utterances from GitHub
import json
import urllib.request

url = 'https://raw.githubusercontent.com/Das-rebel/chuckle-audio-notebooks/main/utterances_clean.jsonl'
print('Downloading utterances...')
urllib.request.urlretrieve(url, '/content/utterances.jsonl')

utterances = []
with open('/content/utterances.jsonl') as f:
    for line in f:
        utterances.append(json.loads(line))
print(f'Loaded {len(utterances)} utterances')

In [ ]:
# Map video_id to audio file
import os

audio_paths = {}
for folder in ['audio', 'audio_all', 'audio_final', 'audio_new']:
    path = f'/content/gdrive/MyDrive/chuckle_audio_all/{folder}'
    if os.path.exists(path):
        for f in os.listdir(path):
            if f.endswith(('.mp3', '.wav')):
                vid = f.rsplit('.', 1)[0]
                audio_paths[vid] = os.path.join(path, f)
print(f'Audio files: {len(audio_paths)}')

In [ ]:
# Filter utterances with audio
available = [u for u in utterances if u['video_id'] in audio_paths]
print(f'Utterances with audio: {len(available)}')

In [ ]:
# Load WavLM
from transformers import WavLMModel
import torch

print('Loading WavLM...')
model = WavLMModel.from_pretrained('microsoft/wavlm-base')
model.eval()
print('WavLM loaded!')

In [ ]:
# Extract embeddings
import librosa
import numpy as np
from tqdm import tqdm

CHECKPOINT_FILE = '/content/gdrive/MyDrive/wavlm_utterance_checkpoint.json'
OUTPUT_FILE = '/content/gdrive/MyDrive/wavlm_utterance_embeddings.jsonl'

# Load checkpoint
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        checkpoint = json.load(f)
    done = set(checkpoint.get('done', []))
else:
    done = set()

print(f'Already done: {len(done)}')

results = []
for i, u in enumerate(tqdm(available)):
    uid = f"{u['video_id']}_{u['start']}"
    if uid in done:
        continue
    
    try:
        audio_path = audio_paths[u['video_id']]
        audio, sr = librosa.load(audio_path, sr=16000, offset=u['start'], duration=u['end']-u['start'])
        if len(audio) < 320:
            continue
        
        with torch.no_grad():
            inputs = torch.FloatTensor(audio).unsqueeze(0)
            outputs = model(inputs)
            emb = outputs.last_hidden_state.mean(1).squeeze().numpy()
        
        results.append({
            'uid': uid,
            'video_id': u['video_id'],
            'embedding': emb.tolist(),
            'label': u.get('label', 0),
            'text': u.get('text', '')[:200]
        })
        done.add(uid)
    except Exception as e:
        continue
    
    if len(results) >= 100:
        with open(OUTPUT_FILE, 'a') as f:
            for r in results:
                f.write(json.dumps(r) + '\n')
        results = []
        
        with open(CHECKPOINT_FILE, 'w') as f:
            json.dump({'done': list(done)}, f)

if results:
    with open(OUTPUT_FILE, 'a') as f:
        for r in results:
            f.write(json.dumps(r) + '\n')

with open(CHECKPOINT_FILE, 'w') as f:
    json.dump({'done': list(done), 'complete': True}, f, indent=2)

print(f'\nDone! Total: {len(done)}')